# 05 · Build the SQLite database

Loads the clean CSVs into `data_clean/analyst_market.db` for the SQL analysis in `sql/`,
creates the summary view used by the Power BI quadrant chart, and exports that view to CSV.

Naming: `fact_` tables hold measurements, `dim_` tables hold descriptions, `v_` = view.

In [ ]:
import sqlite3

import pandas as pd

In [ ]:
panel = pd.read_csv("../data_clean/municipality_panel_2020_2024.csv", dtype={"municipality": str})
ads = pd.read_csv("../data_clean/analyst_ads_final.csv", dtype={"municipality": str})
dim = pd.read_csv("../data_clean/dim_municipality.csv", dtype={"municipality": str, "region_id": str})

## Load tables (re-runnable: `if_exists="replace"`)

In [ ]:
conn = sqlite3.connect("../data_clean/analyst_market.db")

panel.to_sql("fact_panel", conn, if_exists="replace", index=False)
ads.to_sql("fact_ads", conn, if_exists="replace", index=False)
dim.to_sql("dim_municipality", conn, if_exists="replace", index=False)

for table in ["fact_panel", "fact_ads", "dim_municipality"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn)["n"][0]
    print(f"{table:17} {n:,}")    # 1,450 · 6,801 · 290

## Summary view: one row per municipality

Ads are summed over all five years; unemployment, income and rent use 2024 only.
`MAX(CASE WHEN year = 2024 THEN … END)` pivots one year out of five rows (MAX ignores the NULLs).
Same SQL as `sql/06_summary_view.sql`.

In [ ]:
VIEW_SQL = """
DROP VIEW IF EXISTS v_municipality_summary;

CREATE VIEW v_municipality_summary AS
SELECT
    d.municipality,
    d.municipality_name,
    d.region_name,
    ROUND(AVG(p.population)) AS avg_population,
    SUM(p.analyst_ads) AS ads_2020_2024,
    ROUND(SUM(p.analyst_ads) * 10000.0 / AVG(p.population) / 5, 2) AS ads_per_10k_per_year,
    MAX(CASE WHEN p.year = 2024 THEN p.unemployment_rate END) AS unemployment_2024,
    MAX(CASE WHEN p.year = 2024 THEN p.median_income_tkr END) AS income_2024_tkr,
    MAX(CASE WHEN p.year = 2024 THEN p.median_rent_sqm END) AS rent_sqm_2024,
    MAX(CASE WHEN p.year = 2024 THEN p.rent_moe END) AS rent_moe_2024,
    ROUND(100.0 * MAX(CASE WHEN p.year = 2024 THEN p.median_rent_sqm END) * 70
          / (MAX(CASE WHEN p.year = 2024 THEN p.median_income_tkr END) * 1000), 1) AS rent_pct_of_income_2024
FROM fact_panel AS p
JOIN dim_municipality AS d
    ON p.municipality = d.municipality
GROUP BY d.municipality, d.municipality_name, d.region_name;
"""

conn.executescript(VIEW_SQL)
conn.commit()

## Export the view for Power BI

In [ ]:
summary = pd.read_sql("SELECT * FROM v_municipality_summary", conn, dtype={"municipality": str})
summary.to_csv("../data_clean/municipality_summary.csv", index=False)
print(summary.shape)    # (290, 11)

conn.close()            # release the file so DB Browser can open it